In [1]:
import gymnasium as gym
import tensorboard
import optuna
import torch
import torch.nn as nn
import os, json
import numpy as np
import matplotlib.pyplot as plt

from typing import Any
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler

from envs.manipulation import Manipulation
from envs.env_creation import MakeEnv
from ray.tune.registry import register_env

/home/sudhishp/ROS2_MPC+DRL_Manipulation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-07-21 16:54:47,100	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-07-21 16:54:47,196	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
from envs.env_creation import MakeEnv
import inspect
import os

print(inspect.getfile(MakeEnv))

print(os.getcwd())

/home/sudhishp/ROS2_MPC+DRL_Manipulation/.venv/lib/python3.12/site-packages/envs/env_creation.py
/home/sudhishp/ROS2_MPC+DRL_Manipulation/python/envs


In [3]:
%cd /home/sudhishp/ROS2_MPC+DRL_Manipulation

env = Manipulation(
    json_file="/home/sudhishp/ROS2_MPC+DRL_Manipulation/python/envs/environment_params.json",
    render_mode="human",   # use None if you don't need any rendering
    n_obstacles=3,
    max_episode_steps=200,
)


print("model nq:", env.model.nq, " nu:", env.model.nu)
print("action_space:", env.action_space)
print("observation_space:", env.observation_space)


/home/sudhishp/ROS2_MPC+DRL_Manipulation
NMPC compiled - decision vars: 72
model nq: 6  nu: 6
action_space: Box(0.0, 1.0, (10,), float32)
observation_space: Box([-0.8  -0.8  -0.8  -3.05 -1.57 -1.57 -1.57 -3.05 -1.57 -4.   -4.   -4.
 -4.   -4.   -4.    0.  ], [0.8  0.8  0.8  3.05 1.57 1.57 1.57 3.05 1.57 4.   4.   4.   4.   4.
 4.   2.  ], (16,), float32)


`set_state`

Manually pushing a kow qpos/qvel and confirm `_get_obs` reflects it (via `mj_forward`)

In [4]:
qpos = env.init_qpos.copy()
qvel = env.init_qvel.copy()
qpos[:6] = np.array([0.3, -0.2, 0.1, 0.0,0.2, -0.1])

env.set_state(qpos, qvel)
ob = env._get_obs()
print("q from obs (should ~match qpos[:6]):", ob[3:9])
print("qpos[:6] set:                        ", qpos[:6])

q from obs (should ~match qpos[:6]): [ 0.3 -0.2  0.1  0.   0.2 -0.1]
qpos[:6] set:                         [ 0.3 -0.2  0.1  0.   0.2 -0.1]


`_sample_obstacle_positions`

Testig the sampling of obstacle positions with a fixed target, checking the rejection-sampling constraints (min distance to target, min separation between obstacles) hold.

In [5]:
target = np.array([0.25, 0.0, 0.25])
positions = env._sample_obstacle_positions(target_pos=target, n=5, min_target_dist=0.08, min_obs_sep=0.10)

for i, p in enumerate(positions):
    d_target = np.linalg.norm(p - target)
    print(f"obstacle {i}: pos={p}, dist_to_target={d_target:.3f}")

# check pairwise separation
ok = True
for i in range(len(positions)):
    for j in range(i+1, len(positions)):
        d = np.linalg.norm(positions[i]-positions[j])
        if d < 0.10 - 1e-6:
            ok = False
            print(f"  ! obstacles {i},{j} too close: {d:.3f}")
print("All pairwise separations respected:" , ok)


obstacle 0: pos=[0.17668743 0.1954459  0.20710542], dist_to_target=0.213
obstacle 1: pos=[ 0.23396793 -0.1864746   0.28825506], dist_to_target=0.191
obstacle 2: pos=[ 0.34759706 -0.05067352  0.10435608], dist_to_target=0.182
obstacle 3: pos=[0.1175536  0.01631915 0.16553869], dist_to_target=0.158
obstacle 4: pos=[0.15070632 0.01160976 0.34755356], dist_to_target=0.140
All pairwise separations respected: True


`_segment_sphere_distance`

Capsule vs. Sphere Clearance Geometry test

In [6]:
# Case 1: Sphere directly on the segment's midpoint 
T1 = np.array([0.0, 0.0, 0.0])
T2 = np.array([1.0, 0.0, 0.0])
T_on_axis = np.array([0.5, 0.0, 0.0])
D, r = 0.1, 0.05
d = env._segment_sphere_distance(T1, T2,  T_on_axis, D, r)
expected = 0.0 - D/2 - r
print("on-axis midpoint:", d, "expected", expected)

# Case 2: Spehere Directly above the segmnet midpoint at height h -> dist_to_axis = h
h = 0.5
T_above = np.array([0.5, 0.0, h])
d2 = env._segment_sphere_distance(T1, T2, T_above, D, r)
expected = h - D/2 - r
print("on-axis midpoint:", d2, "expected", expected)

# Case 3: sphere beyond the segment endpoint -> falls back to closer-endpoint branch
T_beyond = np.array([2.0, 0.0, 0.0])
d3 = env._segment_sphere_distance(T1, T2, T_beyond, D, r)
expected3 = np.linalg.norm(T_beyond - T2) - D/2 - r
print("beyond endpoint:", d3, " expected:", expected3)

on-axis midpoint: -0.1 expected -0.1
on-axis midpoint: 0.4 expected 0.4
beyond endpoint: 0.8999999999999999  expected: 0.8999999999999999


`_compute_link_obstacle_distances`

Use the live MuJoCo body positions (`self.data.xpos`) for each link segment and the current obstacles.

In [7]:
g_hat = env._compute_link_obstacle_distances()
print("per-link min distance to nearest obstacle:")
for (b1, b2, d), g in zip(
    [("1_Link","2_Link",0.070), ("2_Link","3_Link",0.060), ("3_Link","4_Link",0.056),
     ("4_Link","5_Link",0.050), ("5_Link","6_Link",0.040), ("6_Link","jiazhua_Link",0.040)],
    g_hat
):
    print(f"  {b1:>14} -> {b2:<14}: {g:.4f}")

print("\nmin over all links (this feeds obs[-1] / nearest_obstacle):", g_hat.min())


per-link min distance to nearest obstacle:
          1_Link -> 2_Link        : 0.1189
          2_Link -> 3_Link        : 0.1244
          3_Link -> 4_Link        : 0.1589
          4_Link -> 5_Link        : 0.2242
          5_Link -> 6_Link        : 0.2519
          6_Link -> jiazhua_Link  : inf

min over all links (this feeds obs[-1] / nearest_obstacle): 0.11887095044955204


`_compute_reward`

Testing the 5 reward terms across the branches: normal step, near_goal (r2 bonus), and near/at collision (r3). Called directly with synthetic inputs so we don't need a full NMPC solve to see the reward shape.

In [8]:
env.step_count = 0
env._qdot_norm_prev = 0.0

# Case A: normal step, error shrinking, all links safe
r_a = env._compute_reward(pos_err_norm = 0.20, poss_err_norm_prev=0.25, qdot=np.array([0.1]*6), g_hat = np.full(6, 0.3))
print('normal step reward:', r_a)

# Case B: goal reached (pos_err_norm < 0.03 triggers r2 = -500 inside _compute_reward,
#         though step() actually short-circuits to rew_target_scale before calling this)
r_b = env._compute_reward(pos_err_norm=0.02, poss_err_norm_prev=0.05, qdot=np.array([0.0]*6), g_hat=np.full(6, 0.3))
print("near-goal reward (r2 branch):", r_b)

# Case C: an active collision on one link (g <= 0)
r_c = env._compute_reward(pos_err_norm=0.20, poss_err_norm_prev=0.20, qdot=np.array([0.1]*6), g_hat=np.array([0.3,0.3,0.3,0.3,0.3,-0.01]))
print("collision-on-one-link reward:", r_c)

normal step reward: 24494642.029081777
near-goal reward (r2 branch): -2105.324595503189
collision-on-one-link reward: 24495892.927831784


`step`


Full loop: unpack action -> `nmpc.set_drl_params` -> `NMPCController.solve` (IPOPT) -> `mj_step` -> reward.
This is the most expensive method to test (runs an NLP solve), so start with a single step.

In [9]:
env.reset(seed=1)
action = env.action_space.sample()
print("sampled action (theta_s | theta_r | theta_g):", action)

nobs, rew, term, truncated, info = env.step(action)
print("\nnext obs:", nobs)
print("reward:", rew)
print("terminated:", term, " truncated:", truncated)
print("info:", info)
print("NMPC solver status:", env.nmpc.theta_s, env.nmpc.theta_r, env.nmpc.theta_g)


sampled action (theta_s | theta_r | theta_g): [0.27632186 0.7745841  0.29750848 0.6347918  0.1988517  0.7228524
 0.7162562  0.41387185 0.6825941  0.43295303]

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************


next obs: [ 0.30333745  0.02130789 -0.18347831  0.00212856  0.09012815 -0.07078017
  0.08904253 -0.03742313 -0.01519054 -0.09379622  0.01400478  0.15455282
 -0.2738225   0.08391982  0.05733333  0.13411771]
reward: 0.0355494890292386
terminated: False  truncated: False
info: {}
NMPC solver status: [0.27632186 0.7745841  0.29750848] [0.6347918  0.1988517  0.7228524  0.7162562  0.41387185 0.6825941 ] [0.43295303]


In [10]:
frame = env.render()
if frame is not None:
    print("frame shape:", frame.shape, frame.dtype)
    plt.imshow(frame)
    plt.axis("off")
else:
    print("mujoco_renderer is None (render_mode may be None, or renderer only builds under 'human' mode)")


mujoco_renderer is None (render_mode may be None, or renderer only builds under 'human' mode)
